# Ejercicios 1 y 2 - Conexión y adquisición de Sentinel-2

Este cuaderno documenta la conexión reproducible con Copernicus Data Space mediante openEO y prepara la adquisición de los raster mínimos para los lagos Atitlán y Amatitlán. Se utilizan exclusivamente las 22 fechas oficiales del enunciado.

La ejecución local está separada de las operaciones remotas: por defecto el cuaderno valida configuración, AOI y manifiesto sin autenticar ni crear batch jobs.

## 1. Decisiones de adquisición

- Backend: federación openEO de Copernicus Data Space.
- Colección: Sentinel-2 Level-2A (`SENTINEL2_L2A`).
- Bandas mínimas: B03, B04 y B08 para NDWI/NDVI, más SCL para control de calidad.
- Extensión: cajas oficiales de cada lago en EPSG:4326.
- Tiempo: un intervalo de un día por fecha oficial.
- Salida: GeoTIFF mediante un batch job por lago-fecha.

**Limitación:** los GeoJSON actuales son rectángulos de consulta construidos con las coordenadas del PDF; no son el contorno del agua y no deben usarse como máscara final del lago.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Abra Jupyter desde la raíz de Laboratorio 4')

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from src.adquisicion import (
    connect_openeo,
    download_scenes,
    inspect_collection,
    prepare_repository,
    query_spec,
    select_scenes,
    validate_manifest,
)
from src.config import (
    BANDAS_MINIMAS,
    LAGOS,
    OPENEO_BACKEND_URL,
    OPENEO_COLLECTION_ID,
)

print(f'Raíz del proyecto: {ROOT}')
print(f'Backend: {OPENEO_BACKEND_URL}')
print(f'Colección: {OPENEO_COLLECTION_ID}')

Raíz del proyecto: D:\Tareas\Data Science\Laboratorio 4
Backend: https://openeofed.dataspace.copernicus.eu
Colección: SENTINEL2_L2A


## 2. Preparación local y manifiesto oficial

El manifiesto es la fuente común para el resto del laboratorio. Una fila representa una combinación lago-fecha; cualquier estado de descarga o advertencia queda registrado sin cambiar la lista oficial.

In [2]:
summary = prepare_repository()
rows = validate_manifest()
manifest = pd.DataFrame(rows)

print(f"Escenas oficiales: {summary['scenes']}")
print(f"Bandas iniciales: {', '.join(summary['bands'])}")
display(manifest[[
    'lago', 'fecha', 'satelite_oficial', 'nubosidad_oficial_pct',
    'cobertura_valida_oficial_pct', 'estado', 'quality_flag'
]])

Escenas oficiales: 22
Bandas iniciales: B03, B04, B08, SCL


,lago,fecha,satelite_oficial,nubosidad_oficial_pct,cobertura_valida_oficial_pct,estado,quality_flag
0,amatitlan,2025-01-28,Sentinel-2B,0.06,,pendiente,pendiente_descarga
1,amatitlan,2025-04-15,Sentinel-2A,0.09,,pendiente,pendiente_descarga
2,amatitlan,2025-04-28,Sentinel-2B,1.03,,pendiente,pendiente_descarga
3,amatitlan,2025-11-24,Sentinel-2B,0.50,,pendiente,pendiente_descarga
4,amatitlan,2026-01-08,Sentinel-2C,0.77,,pendiente,pendiente_descarga
5,amatitlan,2026-02-02,Sentinel-2B,0.39,,pendiente,pendiente_descarga
6,amatitlan,2026-02-07,Sentinel-2C,0.02,57.1,pendiente,cobertura_parcial_oficial
7,amatitlan,2026-03-29,Sentinel-2C,0.01,,pendiente,pendiente_descarga
8,amatitlan,2026-04-13,Sentinel-2B,0.09,,pendiente,pendiente_descarga
9,amatitlan,2026-04-28,Sentinel-2C,4.96,,pendiente,pendiente_descarga


In [3]:
counts = manifest.groupby('lago').size().to_dict()
assert counts == {'amatitlan': 11, 'atitlan': 11}
assert not manifest.duplicated(['lago', 'fecha']).any()
assert set(BANDAS_MINIMAS) == {'B03', 'B04', 'B08', 'SCL'}

partial = manifest.query("lago == 'amatitlan' and fecha == '2026-02-07'")
assert len(partial) == 1
assert partial.iloc[0]['cobertura_valida_oficial_pct'] == '57.1'

print('Validación correcta: 11 fechas por lago, sin duplicados.')
print('La advertencia de cobertura parcial de Amatitlán está preservada.')

Validación correcta: 11 fechas por lago, sin duplicados.
La advertencia de cobertura parcial de Amatitlán está preservada.


## 3. Ejercicio 1 - conexión con openEO

La celda siguiente viene desactivada para que el notebook sea reproducible sin credenciales. Al cambiar `EJECUTAR_CONEXION` a `True`, openEO iniciará el flujo OIDC personal y consultará los metadatos de la colección. No se debe copiar al notebook ningún usuario, contraseña ni token.

In [4]:
EJECUTAR_CONEXION = False

if EJECUTAR_CONEXION:
    connection = connect_openeo(authenticate=True)
    collection = inspect_collection(connection)
    print(f"Conexión autenticada. Colección: {collection.get('id')}")
else:
    print('Conexión remota omitida. Cambie EJECUTAR_CONEXION=True para probar OIDC.')

Conexión remota omitida. Cambie EJECUTAR_CONEXION=True para probar OIDC.


## 4. Ejercicio 2 - consulta mínima de una escena

Antes de enviar un batch job se inspecciona la consulta. La fecha final es el día siguiente porque el intervalo openEO delimita el único día oficial que se desea cargar. No se aplica un filtro que reemplace la fecha por una escena distinta.

In [5]:
scene = select_scenes('amatitlan', '2025-01-28')[0]
spec = query_spec(scene)
pd.Series(spec, name='consulta_openEO').to_frame()

,consulta_openEO
collection_id,SENTINEL2_L2A
spatial_extent,"{'west': -90.638065, 'south': 14.412347, 'east..."
temporal_extent,"[2025-01-28, 2025-01-29]"
bands,"[B03, B04, B08, SCL]"


## 5. Descarga controlada

La prueba de extremo a extremo descarga solamente Amatitlán del 28 de enero de 2025. Al finalizar, el código valida los GeoTIFF y actualiza el manifiesto. Si la carpeta de salida ya contiene archivos, la descarga se detiene para no sobrescribir datos crudos.

El lote de 22 escenas se ejecuta posteriormente desde la terminal con `python src/adquisicion.py download --confirm-batch`, después de revisar esta prueba.

In [6]:
EJECUTAR_DESCARGA_PRUEBA = False

if EJECUTAR_DESCARGA_PRUEBA:
    paths = download_scenes([scene], bands=BANDAS_MINIMAS)
    print('Assets descargados:')
    for path in paths:
        print(f'- {path}')
else:
    print('Descarga omitida. Cambie EJECUTAR_DESCARGA_PRUEBA=True para crear un batch job.')

Descarga omitida. Cambie EJECUTAR_DESCARGA_PRUEBA=True para crear un batch job.


## 6. Resultado de los ejercicios 1 y 2

Quedó definido un flujo único para conectar, consultar y descargar Sentinel-2 sin exponer credenciales. El inventario contiene las 22 fechas oficiales, limita cada consulta a la caja del lago y solicita únicamente B03, B04, B08 y SCL. Los raster originales se conservan sin sobrescritura y su trazabilidad se mantiene en `manifest_escenas.csv`.

La máscara del contorno real del agua y las bandas adicionales del script exacto de cianobacteria quedan explícitamente pendientes para las etapas que dependen de esos insumos; no se sustituyen con el bbox ni con una imagen RGB.